In [2]:
print(df.isna().sum())                    # empty cells per column
print(df.duplicated().sum())              # fully repeated rows
print(df["user id"].duplicated().sum())   # repeated users
print(df["test group"].value_counts())    # group sizes
print(df["converted"].value_counts())     # how many True and False

Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64
0
0
test group
ad     564577
psa     23524
Name: count, dtype: int64
converted
False    573258
True      14843
Name: count, dtype: int64


In [1]:
import pandas as pd

df = pd.read_csv("marketing_AB.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(588101, 7)
['Unnamed: 0', 'user id', 'test group', 'converted', 'total ads', 'most ads day', 'most ads hour']


,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [3]:
summary = df.groupby("test group")["converted"].agg(["count", "sum", "mean"])
summary.columns = ["people", "buyers", "conversion_rate"]
summary["conversion_rate_percent"] = (summary["conversion_rate"] * 100).round(3)
print(summary)

            people  buyers  conversion_rate  conversion_rate_percent
test group                                                          
ad          564577   14423         0.025547                    2.555
psa          23524     420         0.017854                    1.785


In [4]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

ad = df[df["test group"] == "ad"]["converted"]
psa = df[df["test group"] == "psa"]["converted"]

buyers = [ad.sum(), psa.sum()]    # ad first, psa second
people = [len(ad), len(psa)]

z, p = proportions_ztest(buyers, people, alternative="two-sided")
low, high = confint_proportions_2indep(buyers[0], people[0], buyers[1], people[1],
                                       method="wald", compare="diff", alpha=0.05)

print("z statistic:", round(z, 3))
print("p value:", f"{p:.3e}")
print("gap, ad minus psa, in points:", round((ad.mean() - psa.mean()) * 100, 3))
print("95% confidence interval for the gap, in points:", round(low * 100, 3), "to", round(high * 100, 3))

z statistic: 7.37
p value: 1.705e-13
gap, ad minus psa, in points: 0.769
95% confidence interval for the gap, in points: 0.595 to 0.943


In [5]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

day = df.groupby(["most ads day", "test group"])["converted"].agg(
    people="count", buyers="sum", rate="mean").reset_index()
day["rate_percent"] = (day["rate"] * 100).round(3)

table = day.pivot(index="most ads day", columns="test group",
                  values=["people", "rate_percent"]).reindex(order)
table["gap_points"] = (table[("rate_percent", "ad")] - table[("rate_percent", "psa")]).round(3)

print(table)

               people         rate_percent        gap_points
test group         ad     psa           ad    psa           
most ads day                                                
Monday        83571.0  3502.0        3.324  2.256      1.068
Tuesday       74572.0  2907.0        3.044  1.445      1.599
Wednesday     77418.0  3490.0        2.536  1.576      0.960
Thursday      79077.0  3905.0        2.164  2.023      0.141
Friday        88805.0  3803.0        2.246  1.630      0.616
Saturday      78802.0  2858.0        2.131  1.400      0.731
Sunday        82332.0  3059.0        2.462  2.059      0.403
